In [1]:
%%time
import acme4_explore
import datamapplot as dmp
from fast_hdbscan import HDBSCAN
import io
import json
import logging as lg
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
import re
from sentencepiece import SentencePieceTrainer, SentencePieceProcessor
from sklearn.metrics.pairwise import paired_distances
from tqdm.auto import tqdm, trange
import umap
import vectorizers as vz
import vectorizers.transformers as vzt
import zstandard as zstd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
import seaborn as sns
from collections import Counter
import igraph as ig
import pickle
import gzip


CPU times: user 34.1 s, sys: 1.88 s, total: 36 s
Wall time: 25.5 s


In [2]:
## import data
with gzip.open('pt_data.pkl.gz', 'rb') as fp:
    process_df, Trees = pickle.load(fp)

## revisiting -- process tree clustering via bag of features

We consider subtrees labelled with the root's process name.

We build bag of features made up of every process in the tree, by concatenating the following:

* process name
* layer (distance from root)
* counter (useful to consider multiple occurrences of the same process at the same layer)


In [3]:
## compute bag of features
def subtree_features(sg):
    root = np.where(np.array(sg.degree(mode='in'))==0)[0][0]
    V, L, _ = sg.bfs(root)
    F = []
    for i in range(len(L)-1):
        P = []
        for j in np.arange(L[i],L[i+1]):
            P.append(sg.vs[V[j]]['process'])
        C = Counter(P)
        keys = list(C.keys())
        vals = list(C.values())
        for j in range(len(keys)):
            for k in range(vals[j]):
                F.append(str(i)+'::'+keys[j]+'::'+str(k)) 
    return set(F)

## overlap coefficient (aka Szymkiewicz-Simpson)
def my_sim(a,b):
    return len(a.intersection(b))/min(len(a),len(b))

## utility function -- getting a subtree from Trees object
def get_subtree(Trees, tree_id, root_id):
    nodes, _ ,_ = Trees.subgraph(tree_id).bfs(root_id)
    g = Trees.subgraph(tree_id).subgraph(nodes)
    g["ly"] = g.layout_reingold_tilford() ## save a layout for plotting
    return g

In [4]:
## example of a subtree and its features
sg = get_subtree(Trees, 731, 132)
subtree_features(sg)


{'0::smss.exe::0',
 '1::csrss.exe::0',
 '1::winlogon.exe::0',
 '2::dwm.exe::0',
 '2::fontdrvhost.exe::0',
 '2::logonui.exe::0',
 '2::userinit.exe::0',
 '2::wlrmdr.exe::0',
 '3::explorer.exe::0',
 '4::cmd.exe::0',
 '4::cmd.exe::1',
 '4::securityhealthsystray.exe::0',
 '5::conhost.exe::0',
 '5::conhost.exe::1',
 '5::ec2launch.exe::0',
 '5::notepad.exe::0'}

In [5]:
with open('pt_features.pkl','rb') as fp:
    Features = pickle.load(fp)


In [35]:
## consider the most common root process names
P = list(process_df['process'])
Counter(P).most_common(25)


[('bash.exe', 491),
 ('git.exe', 355),
 ('ngentask.exe', 304),
 ('cmd.exe', 241),
 ('taskhostw.exe', 151),
 ('compattelrunner.exe', 149),
 ('smss.exe', 94),
 ('winlogon.exe', 89),
 ('ssm-document-worker.exe', 84),
 ('explorer.exe', 44),
 ('userinit.exe', 43),
 ('ntoskrnl.exe', 41),
 ('python.exe', 41),
 ('setup.exe', 39),
 ('svchost.exe', 39),
 ('conhost.exe', 36),
 ('powershell.exe', 32),
 ('msedge.exe', 27),
 ('aws.exe', 24),
 ('code.exe', 23),
 ('firefox.exe', 22),
 ('ngen.exe', 20),
 ('py.exe', 17),
 ('downloadfroms3.exe', 16),
 ('microsoftedgeupdate.exe', 15)]

In [36]:
top_process = 17
mc = set([x[0] for x in Counter(P).most_common(top_process)])
P_subset = [p for p in P if p in mc]
F_subset = [f for (p,f) in zip(P,Features) if p in mc]
B = list(process_df['has_bad'])
B_subset = [b for (p,b) in zip(P,B) if p in mc]

## compute similarities
l = len(F_subset)
S = np.zeros(shape=(l,l))
for i in range(len(F_subset)-1):
    for j in np.arange(i+1,len(F_subset)):
        s = my_sim(F_subset[i],F_subset[j])
        S[i,j] = S[j,i] = s


In [37]:
## cluster - set the correct number of clusters
from sklearn.cluster import AgglomerativeClustering
model = AgglomerativeClustering(n_clusters=top_process, metric='precomputed', linkage='complete')
model.fit(1-S)


,n_clusters,17
,metric,'precomputed'
,memory,None
,connectivity,None
,compute_full_tree,'auto'
,linkage,'complete'
,distance_threshold,None
,compute_distances,False


In [38]:
## list cluster compositions
for s in range(top_process):
    print( Counter([P_subset[i] for i in np.where(model.labels_==s)[0]]) , 'with bad:', Counter([B_subset[i] for i in np.where(model.labels_==s)[0]]))


Counter({'bash.exe': 489, 'git.exe': 355, 'compattelrunner.exe': 149, 'ssm-document-worker.exe': 84, 'powershell.exe': 18, 'python.exe': 14, 'cmd.exe': 14, 'svchost.exe': 11, 'explorer.exe': 5, 'userinit.exe': 2, 'setup.exe': 1, 'ntoskrnl.exe': 1}) with bad: Counter({0: 1138, 1: 5})
Counter({'smss.exe': 83, 'conhost.exe': 9, 'svchost.exe': 1}) with bad: Counter({0: 83, 1: 10})
Counter({'ntoskrnl.exe': 38}) with bad: Counter({0: 36, 1: 2})
Counter({'ngentask.exe': 304, 'powershell.exe': 5, 'cmd.exe': 4}) with bad: Counter({0: 311, 1: 2})
Counter({'python.exe': 27, 'explorer.exe': 26, 'conhost.exe': 5, 'cmd.exe': 3, 'powershell.exe': 2, 'bash.exe': 2}) with bad: Counter({0: 59, 1: 6})
Counter({'taskhostw.exe': 151, 'setup.exe': 4, 'explorer.exe': 4, 'powershell.exe': 3, 'cmd.exe': 2}) with bad: Counter({0: 161, 1: 3})
Counter({'smss.exe': 11, 'ntoskrnl.exe': 2}) with bad: Counter({0: 9, 1: 4})
Counter({'userinit.exe': 9, 'conhost.exe': 2, 'cmd.exe': 2}) with bad: Counter({0: 11, 1: 2})
C

In [13]:
### try with HDBSCAN and look at points not clustered

In [49]:
from sklearn.cluster import HDBSCAN
model = HDBSCAN(metric='precomputed', min_cluster_size=15)
model.fit(1-S)


,min_cluster_size,15
,min_samples,None
,cluster_selection_epsilon,0.0
,max_cluster_size,None
,metric,'precomputed'
,metric_params,None
,alpha,1.0
,algorithm,'auto'
,leaf_size,40
,n_jobs,None
,cluster_selection_method,'eom'


In [50]:
for s in np.arange(-1,max(model.labels_)+1):
    print(s, Counter([P_subset[i] for i in np.where(model.labels_==s)[0]]), 'with bad:', Counter([B_subset[i] for i in np.where(model.labels_==s)[0]]))


-1 Counter({'ntoskrnl.exe': 29, 'python.exe': 26, 'svchost.exe': 18, 'userinit.exe': 12, 'smss.exe': 12, 'bash.exe': 11, 'cmd.exe': 10, 'explorer.exe': 9, 'powershell.exe': 9, 'conhost.exe': 5, 'setup.exe': 4}) with bad: Counter({0: 122, 1: 23})
0 Counter({'cmd.exe': 26, 'conhost.exe': 12, 'svchost.exe': 10, 'powershell.exe': 8, 'setup.exe': 1}) with bad: Counter({0: 57})
1 Counter({'git.exe': 355, 'bash.exe': 6, 'svchost.exe': 3, 'powershell.exe': 2, 'setup.exe': 1, 'conhost.exe': 1}) with bad: Counter({0: 364, 1: 4})
2 Counter({'taskhostw.exe': 151, 'powershell.exe': 2, 'conhost.exe': 2, 'ntoskrnl.exe': 1, 'svchost.exe': 1}) with bad: Counter({0: 157})
3 Counter({'smss.exe': 7, 'powershell.exe': 5, 'cmd.exe': 4, 'setup.exe': 3, 'python.exe': 3, 'svchost.exe': 3, 'userinit.exe': 2, 'conhost.exe': 1, 'ntoskrnl.exe': 1}) with bad: Counter({0: 27, 1: 2})
4 Counter({'winlogon.exe': 89}) with bad: Counter({0: 81, 1: 8})
5 Counter({'smss.exe': 75, 'conhost.exe': 9, 'svchost.exe': 1}) with b

In [41]:
Counter(B_subset)

Counter({0: 2206, 1: 67})

## Where to next???

* edge features (below) does not work
* 3-node paths? consider all nodes with some in AND out degree, link then all

### simple bag of edge features


In [52]:
## compute bag of features
def subtree_edge_features(sg):
    P = sg.vs['process']
    return set([ P[e.source]+'::'+P[e.target] for e in sg.es ])


In [53]:
Features = []
for t,r in zip(process_df.tree, process_df.root):
    sg = get_subtree(Trees, t, r)
    Features.append(subtree_edge_features(sg))


In [54]:
## consider the most common root process names
P = list(process_df['process'])
top_process = 9
mc = set([x[0] for x in Counter(P).most_common(top_process)])
P_subset = [p for p in P if p in mc]
F_subset = [f for (p,f) in zip(P,Features) if p in mc]

## compute similarities
l = len(F_subset)
S = np.zeros(shape=(l,l))
for i in range(len(F_subset)-1):
    for j in np.arange(i+1,len(F_subset)):
        s = my_sim(F_subset[i],F_subset[j])
        S[i,j] = S[j,i] = s


In [55]:
model = HDBSCAN(metric='precomputed', min_cluster_size=10)
model.fit(1-S)


,min_cluster_size,10
,min_samples,None
,cluster_selection_epsilon,0.0
,max_cluster_size,None
,metric,'precomputed'
,metric_params,None
,alpha,1.0
,algorithm,'auto'
,leaf_size,40
,n_jobs,None
,cluster_selection_method,'eom'


In [56]:
for s in range(max(model.labels_)):
    print(Counter([P_subset[i] for i in np.where(model.labels_==s)[0]]))


Counter({'cmd.exe': 13})
Counter({'bash.exe': 9, 'cmd.exe': 1})
Counter({'bash.exe': 468, 'git.exe': 355, 'ngentask.exe': 304, 'cmd.exe': 206, 'taskhostw.exe': 151, 'compattelrunner.exe': 149, 'smss.exe': 94, 'winlogon.exe': 89, 'ssm-document-worker.exe': 84})


In [57]:
Counter(model.labels_)

Counter({np.int64(2): 1900,
         np.int64(3): 19,
         np.int64(-1): 16,
         np.int64(0): 13,
         np.int64(1): 10})

## nearest neighbours with badusers as labels


In [58]:
## compute similarities
l = len(Features)
S = np.zeros(shape=(l,l))
for i in range(len(Features)-1):
    for j in np.arange(i+1,len(Features)):
        s = my_sim(Features[i],Features[j])
        S[i,j] = S[j,i] = s


In [59]:
from sklearn.neighbors import NearestNeighbors as NN
model = NN(n_neighbors=11, metric='precomputed')
model.fit(1-S)


,n_neighbors,11
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'precomputed'
,p,2
,metric_params,None
,n_jobs,None


In [60]:
## size of intersections - NNs and 
_, indices = model.kneighbors()
Bad = set(np.where(process_df.has_bad)[0])
bad_nn = [len(set(v).intersection(Bad)) for v in indices]

## average number of NN trees with baduser
_temp = pd.DataFrame(np.array([list(process_df.has_bad) , bad_nn]).T,columns=['bad','bad_nn'])
_temp.groupby(by='bad').mean()


,bad_nn
bad,
0,0.217408
1,4.236641


### Quick EDA

Top non-baduser trees with several baduser NNs have the same shape, a sequence of three 'wsl.exe' processes.
    

In [62]:
_temp.sort_values(by='bad_nn', ascending=False).head(15)


,bad,bad_nn
863,1,11
850,1,11
840,1,11
2206,1,11
810,1,11
2207,1,11
800,1,11
2208,1,11
788,1,11
722,0,11


In [64]:
list(process_df.process_dict)[1741]

{0: Counter({'default-browser-agent.exe': 1}),
 1: Counter({'firefox.exe': 1}),
 2: Counter({'firefox.exe': 1}),
 3: Counter({'pingsender.exe': 1, 'firefox.exe': 1})}